# Tutorial 53: HuggingFace the native way

The cell below is written exactly the way notebooks use the Hub every day —
a plain `load_dataset("ylecun/mnist")`, no paths, no download declarations.
The `%%krauncher` magic translates the established practice automatically:

1. it detects the literal Hub reference in the cell's AST,
2. sizes it via the Hub API — the pre-run quote gets an honest IO estimate,
3. and pre-fetches the repo into the worker's HF hub cache **before** the
   container starts, so the unmodified `load_dataset()` call finds the data
   through `HF_HOME` — and the download IO lands in the measured download
   phase instead of being billed as compute.

Dynamic references (`from_pretrained(name_variable)`) can't be pre-fetched —
the magic warns that their IO will run inside execution.


In [ ]:
%load_ext krauncher_magic

In [ ]:
%env CAS_CLIENT_CONFIG=../../cas-client/.env

In [ ]:
num_epochs = 1
batch_size = 128

In [ ]:
%%krauncher --pip datasets --timeout 300
import numpy as np
import torch
import torch.nn as nn
from datasets import load_dataset
from torch.utils.data import DataLoader

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

# Native practice: no paths — the pre-fetched hub cache serves this call.
ds = load_dataset("ylecun/mnist")
train_data = ds["train"]
test_data = ds["test"]

def collate(batch):
    images = torch.stack([
        torch.from_numpy(np.array(x["image"], dtype=np.float32)).unsqueeze(0) / 255.0
        for x in batch
    ])
    labels = torch.tensor([x["label"] for x in batch])
    return images, labels

train_loader = DataLoader(train_data, batch_size=batch_size, shuffle=True, collate_fn=collate)
test_loader = DataLoader(test_data, batch_size=batch_size, collate_fn=collate)

model = nn.Sequential(
    nn.Conv2d(1, 32, 3, padding=1),
    nn.ReLU(),
    nn.MaxPool2d(2),
    nn.Conv2d(32, 64, 3, padding=1),
    nn.ReLU(),
    nn.MaxPool2d(2),
    nn.Flatten(),
    nn.Linear(64 * 7 * 7, 128),
    nn.ReLU(),
    nn.Linear(128, 10),
).to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss()

model.train()
for epoch in range(num_epochs):
    total_loss = 0.0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        loss = criterion(model(images), labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f"Epoch {epoch + 1}/{num_epochs}, loss={total_loss / len(train_loader):.4f}")

model.eval()
correct = total = 0
with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        preds = model(images).argmax(dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

accuracy = correct / total
train_samples = len(train_data)
test_samples = len(test_data)
print(f"Test accuracy: {accuracy:.4f}")


In [ ]:
print(f"accuracy={accuracy:.4f} ({train_samples} train / {test_samples} test)")